# Local Setup

This notebook prepares your local environment for the Kafka notebooks.

**Prerequisites** — start the local cluster first:

```bash
docker compose up -d
```

Wait about 15 seconds for Kafka to be ready, then run the cells below in order.

## Install required libraries

In [ ]:
%%bash
pip install kafka-python requests psycopg2-binary

## Create local config

Creates `config/kafka_config.py` with connection details for the local Docker environment.

The `kafka_connection` dict auto-selects `PLAINTEXT` (local) or `SSL` (if certificate files are present in `kafkacerts/`), so the same config works with a secured remote cluster too.

In [ ]:
import os

os.makedirs('config', exist_ok=True)
open('config/__init__.py', 'w').close()

cfg = [
    'hostname = "kafka"',
    'port = 9092',
    'cert_folder = "kafkacerts"',
    'topic_name = "pizzas"',
    'pg_user = "postgres"',
    'pg_pwd = "postgres"',
    'timeout_ms = 5000',
    '',
    'import os as _os',
    'kafka_connection = {',
    '    "bootstrap_servers": f"{hostname}:{port}",',
    '    "security_protocol": "SSL" if _os.path.exists(f"{cert_folder}/ca.pem") else "PLAINTEXT",',
    '}',
    'if kafka_connection["security_protocol"] == "SSL":',
    '    kafka_connection["ssl_cafile"] = f"{cert_folder}/ca.pem"',
    '    kafka_connection["ssl_certfile"] = f"{cert_folder}/service.cert"',
    '    kafka_connection["ssl_keyfile"] = f"{cert_folder}/service.key"',
]

with open('config/kafka_config.py', 'w') as f:
    f.write('\n'.join(cfg) + '\n')

print('config/kafka_config.py created')

## Verify connection

Confirm the broker is reachable and list any existing topics.

In [ ]:
from kafka import KafkaConsumer
from config.kafka_config import *

consumer = KafkaConsumer(**kafka_connection)
topics = consumer.topics()
consumer.close()

print('Connected! Available topics:', topics or '(none yet — that is fine)')